# Building Interactive Reports with Pandas and Matplotlib

This notebook builds on the foundations from the previous two tutorials. Here we explore more advanced techniques: generating multiple plots programmatically using loops, modifying DataFrames by adding and removing rows and columns, using pandas' built-in plotting capabilities, creating interactive elements with ipywidgets, and assembling saved figures into a cohesive report.

The ability to automate repetitive visualisation tasks becomes essential when working with real datasets. If you need to create the same type of chart for twenty different countries or fifty different variables, writing the plotting code twenty or fifty times is tedious and error-prone. Loops solve this elegantly. Similarly, interactive widgets let you or your audience explore data without modifying code, which is valuable both for your own exploration and for sharing notebooks with others who may not be comfortable editing Python.

We continue with environmental data throughout, and the final section demonstrates how to assemble multiple saved figures into a simple report structure.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os

%matplotlib inline

# Create a directory for saved figures
os.makedirs('report_figures', exist_ok=True)

## Working Dataset

Let's create a more comprehensive dataset that we'll use throughout this notebook. This represents annual environmental indicators for several countries over multiple years.

In [ ]:
# Create a multi-year, multi-country dataset
np.random.seed(42)

countries = ['Germany', 'France', 'UK', 'Spain', 'Italy', 'Poland', 'Netherlands', 'Sweden']
years = list(range(2015, 2024))

# Generate data with realistic patterns
data_rows = []

# Base values for each country (will vary by year)
base_renewable = {'Germany': 30, 'France': 20, 'UK': 25, 'Spain': 35, 
                  'Italy': 35, 'Poland': 12, 'Netherlands': 12, 'Sweden': 55}
base_emissions = {'Germany': 800, 'France': 300, 'UK': 350, 'Spain': 250,
                  'Italy': 320, 'Poland': 310, 'Netherlands': 150, 'Sweden': 40}

for country in countries:
    for i, year in enumerate(years):
        # Renewables trend upward over time
        renewable = base_renewable[country] + i * 2.5 + np.random.normal(0, 2)
        renewable = min(renewable, 85)  # Cap at realistic maximum
        
        # Emissions trend downward (mostly)
        emissions = base_emissions[country] - i * 8 + np.random.normal(0, 15)
        emissions = max(emissions, 30)  # Floor at realistic minimum
        
        # Energy consumption varies somewhat randomly
        consumption = 200 + len(country) * 20 + np.random.normal(0, 30)
        
        data_rows.append({
            'country': country,
            'year': year,
            'renewable_percent': round(renewable, 1),
            'co2_emissions_mt': round(emissions, 1),
            'energy_consumption_twh': round(consumption, 1)
        })

df = pd.DataFrame(data_rows)
print(f"Dataset shape: {df.shape}")
df.head(15)

## Modifying DataFrames: Adding and Removing Data

Real data analysis often requires modifying your DataFrame as you work. You might need to add new calculated columns, remove irrelevant ones, add new observations, or filter out problematic rows. Understanding these operations gives you flexibility in preparing data for analysis.

### Adding Columns

We covered basic column creation in the first notebook. Here are additional techniques for more complex scenarios.

In [ ]:
# Simple calculated column (review from notebook 1)
df['emissions_per_twh'] = df['co2_emissions_mt'] / df['energy_consumption_twh']

df.head()

In [ ]:
# Adding a column based on conditions using np.where()
# np.where(condition, value_if_true, value_if_false)

df['high_renewable'] = np.where(df['renewable_percent'] > 40, 'Yes', 'No')

df[df['year'] == 2023][['country', 'renewable_percent', 'high_renewable']]

In [ ]:
# For more complex conditions, np.select() is useful
# It takes a list of conditions and a list of corresponding values

conditions = [
    df['renewable_percent'] >= 50,
    df['renewable_percent'] >= 30,
    df['renewable_percent'] >= 15
]
labels = ['Leader', 'Progressing', 'Developing']

df['renewable_category'] = np.select(conditions, labels, default='Lagging')

# Check the distribution of categories
df['renewable_category'].value_counts()

The `np.select()` function evaluates conditions in order and assigns the first matching label. This is useful for creating ordinal categories based on numeric thresholds.

### Removing Columns

The `drop()` method removes columns (or rows). For columns, you specify `axis=1`.

In [ ]:
# Remove the high_renewable column (we'll keep renewable_category)
df = df.drop(columns=['high_renewable'])

# Alternatively: df = df.drop('high_renewable', axis=1)

df.columns

In [ ]:
# Remove multiple columns at once
# Let's first add some temporary columns, then remove them

df['temp1'] = 0
df['temp2'] = 0

print("Before drop:", list(df.columns))

df = df.drop(columns=['temp1', 'temp2'])

print("After drop:", list(df.columns))

### Adding Rows

New rows can be added using `pd.concat()` to combine DataFrames. The older `append()` method has been deprecated in recent pandas versions.

In [ ]:
# Create a new row as a DataFrame
new_country_data = pd.DataFrame([{
    'country': 'Denmark',
    'year': 2023,
    'renewable_percent': 84.0,
    'co2_emissions_mt': 25.0,
    'energy_consumption_twh': 32.5,
    'emissions_per_twh': 0.77,
    'renewable_category': 'Leader'
}])

# Concatenate with the original DataFrame
df = pd.concat([df, new_country_data], ignore_index=True)

# Check that it was added
df[df['country'] == 'Denmark']

In [ ]:
# Adding multiple rows at once
more_denmark = pd.DataFrame([
    {'country': 'Denmark', 'year': 2022, 'renewable_percent': 80.0, 
     'co2_emissions_mt': 28.0, 'energy_consumption_twh': 33.0,
     'emissions_per_twh': 0.85, 'renewable_category': 'Leader'},
    {'country': 'Denmark', 'year': 2021, 'renewable_percent': 75.0,
     'co2_emissions_mt': 30.0, 'energy_consumption_twh': 34.0,
     'emissions_per_twh': 0.88, 'renewable_category': 'Leader'}
])

df = pd.concat([df, more_denmark], ignore_index=True)

df[df['country'] == 'Denmark']

The `ignore_index=True` parameter resets the index to consecutive integers. Without it, you might end up with duplicate index values, which can cause confusion.

### Removing Rows

Rows can be removed by index position, by condition, or by dropping duplicates.

In [ ]:
# Check the current shape
print(f"Current shape: {df.shape}")

# Remove rows by condition (keep only rows that don't match)
# For example, remove all data before 2018
df_recent = df[df['year'] >= 2018].copy()

print(f"After filtering to 2018+: {df_recent.shape}")

In [ ]:
# Remove specific rows by index
# First, let's see the last few rows
print(df.tail())

# Get the index of the last row
last_index = df.index[-1]
print(f"\nLast index: {last_index}")

In [ ]:
# Drop rows by index (removes the Denmark 2021 row we just added)
df = df.drop(index=[last_index])

# Verify
df[df['country'] == 'Denmark']

In [ ]:
# Removing duplicate rows
# Let's first create a duplicate intentionally
duplicate_row = df[df['country'] == 'Denmark'].iloc[[0]]
df = pd.concat([df, duplicate_row], ignore_index=True)

print(f"With duplicate: {df.shape}")
print(f"Denmark rows: {len(df[df['country'] == 'Denmark'])}")

# Remove duplicates (keeps first occurrence by default)
df = df.drop_duplicates()

print(f"After drop_duplicates: {df.shape}")
print(f"Denmark rows: {len(df[df['country'] == 'Denmark'])}")

## Pandas Built-in Plotting

Pandas DataFrames have a `.plot()` method that provides a convenient interface to matplotlib. For quick exploratory visualisation, this is often faster than writing full matplotlib code. The syntax is `df.plot(kind='...', ...)` where kind specifies the chart type.

In [ ]:
# Get data for a single country to demonstrate
germany = df[df['country'] == 'Germany'].sort_values('year')

# Line plot directly from the DataFrame
germany.plot(x='year', y='renewable_percent', kind='line', 
             figsize=(10, 5), title='Germany Renewable Energy %',
             color='green', marker='o')

plt.ylabel('Renewable %')
plt.show()

In [ ]:
# Multiple columns on one plot
germany.plot(x='year', y=['renewable_percent', 'co2_emissions_mt'], 
             kind='line', figsize=(10, 5),
             title='Germany: Renewables vs Emissions',
             secondary_y='co2_emissions_mt')  # Second y-axis for different scale

plt.show()

The `secondary_y` parameter creates a second y-axis on the right side, which is useful when plotting variables with different scales. Without it, the smaller-valued series would appear flat against the larger one.

In [ ]:
# Bar chart from pandas
latest = df[df['year'] == 2023].set_index('country')

latest['renewable_percent'].plot(kind='bar', figsize=(10, 5),
                                  color='teal', edgecolor='white',
                                  title='Renewable Energy % by Country (2023)')

plt.ylabel('Renewable %')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

In [ ]:
# Histogram from pandas
df['renewable_percent'].plot(kind='hist', bins=20, figsize=(10, 5),
                              color='steelblue', edgecolor='white',
                              title='Distribution of Renewable Energy %')

plt.xlabel('Renewable %')
plt.show()

In [ ]:
# Scatter plot
df.plot(kind='scatter', x='renewable_percent', y='co2_emissions_mt',
        figsize=(10, 6), alpha=0.6, color='coral',
        title='Renewable Energy % vs CO2 Emissions')

plt.xlabel('Renewable Energy %')
plt.ylabel('CO2 Emissions (Mt)')
plt.show()

The scatter plot reveals an expected pattern: higher renewable percentages tend to correlate with lower emissions. The spread reflects differences between countries in their overall energy consumption and industrial base.

## Creating Multiple Plots with Loops

When you need to create the same type of visualisation for many different subsets of your data, loops eliminate repetitive code. The pattern is: iterate over your grouping variable, filter the data for each group, and create the plot.

In [ ]:
# Create a line chart for each country showing renewable energy over time

# Get unique countries
countries_in_data = df['country'].unique()
print(f"Countries: {countries_in_data}")

In [ ]:
# Simple loop: one figure per country
for country in countries_in_data[:3]:  # Just first 3 for demonstration
    country_data = df[df['country'] == country].sort_values('year')
    
    plt.figure(figsize=(8, 4))
    plt.plot(country_data['year'], country_data['renewable_percent'], 
             marker='o', linewidth=2)
    plt.xlabel('Year')
    plt.ylabel('Renewable Energy %')
    plt.title(f'{country}: Renewable Energy Trend')
    plt.tight_layout()
    plt.show()

Creating separate figures works, but comparing countries is difficult. A better approach places all countries on one figure as subplots.

In [ ]:
# Multiple subplots in a grid
n_countries = len(countries_in_data)
n_cols = 3
n_rows = (n_countries + n_cols - 1) // n_cols  # Ceiling division

fig, axes = plt.subplots(n_rows, n_cols, figsize=(15, 4 * n_rows), sharey=True)
axes = axes.flatten()  # Convert 2D array to 1D for easier indexing

for i, country in enumerate(countries_in_data):
    country_data = df[df['country'] == country].sort_values('year')
    
    axes[i].plot(country_data['year'], country_data['renewable_percent'],
                 marker='o', linewidth=2, color='teal')
    axes[i].set_title(country)
    axes[i].set_xlabel('Year')
    if i % n_cols == 0:  # Only label y-axis on leftmost plots
        axes[i].set_ylabel('Renewable %')

# Hide any unused subplots
for j in range(i + 1, len(axes)):
    axes[j].set_visible(False)

fig.suptitle('Renewable Energy Trends by Country', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

The `sharey=True` parameter ensures all subplots use the same y-axis scale, making visual comparisons meaningful. The ceiling division `(n + cols - 1) // cols` calculates how many rows we need to fit all countries.

Another common pattern puts all lines on a single plot with a legend.

In [ ]:
# All countries on one plot
plt.figure(figsize=(12, 6))

# Use a colour map to assign distinct colours
colors = plt.cm.tab10(np.linspace(0, 1, len(countries_in_data)))

for i, country in enumerate(countries_in_data):
    country_data = df[df['country'] == country].sort_values('year')
    plt.plot(country_data['year'], country_data['renewable_percent'],
             marker='o', linewidth=2, label=country, color=colors[i])

plt.xlabel('Year', fontsize=12)
plt.ylabel('Renewable Energy %', fontsize=12)
plt.title('Renewable Energy Trends Across Europe', fontsize=14)
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')  # Legend outside plot
plt.tight_layout()
plt.show()

### Saving Multiple Figures in a Loop

For report generation, you often want to save figures to files rather than displaying them. The key change is using `plt.savefig()` instead of or in addition to `plt.show()`.

In [ ]:
# Save individual country plots to files
saved_files = []

for country in countries_in_data:
    country_data = df[df['country'] == country].sort_values('year')
    
    fig, ax = plt.subplots(figsize=(8, 5))
    
    ax.plot(country_data['year'], country_data['renewable_percent'],
            marker='o', linewidth=2, color='teal')
    ax.fill_between(country_data['year'], country_data['renewable_percent'],
                    alpha=0.3, color='teal')
    ax.set_xlabel('Year')
    ax.set_ylabel('Renewable Energy %')
    ax.set_title(f'{country}: Renewable Energy Trend')
    
    # Create a safe filename (replace spaces with underscores)
    safe_name = country.lower().replace(' ', '_')
    filename = f'report_figures/{safe_name}_renewable.png'
    
    fig.savefig(filename, dpi=150, bbox_inches='tight')
    saved_files.append(filename)
    plt.close(fig)  # Close to free memory (important when creating many figures)

print(f"Saved {len(saved_files)} figures:")
for f in saved_files:
    print(f"  {f}")

The `plt.close(fig)` call is important when generating many figures in a loop. Without it, matplotlib keeps all figures in memory, which can cause problems with large numbers of plots.

## Interactive Visualisation with ipywidgets

Jupyter notebooks can include interactive elements that let users explore data without writing code. The ipywidgets library provides controls like sliders, dropdowns, and buttons that can update visualisations dynamically.

This interactivity is valuable in several contexts: when exploring your own data to quickly compare different subsets, when sharing notebooks with colleagues who may not be comfortable modifying Python, and when creating dashboards for presenting findings.

In [ ]:
# Import ipywidgets
import ipywidgets as widgets
from IPython.display import display

### Basic Widget Types

Let's explore the main widget types before connecting them to visualisations.

In [ ]:
# Dropdown widget
country_dropdown = widgets.Dropdown(
    options=list(countries_in_data),
    value='Germany',
    description='Country:'
)

display(country_dropdown)

In [ ]:
# You can read the current value
print(f"Selected country: {country_dropdown.value}")

In [ ]:
# Slider widget
year_slider = widgets.IntSlider(
    value=2020,
    min=2015,
    max=2023,
    step=1,
    description='Year:'
)

display(year_slider)

In [ ]:
# Range slider (select a range of values)
year_range = widgets.IntRangeSlider(
    value=[2018, 2023],
    min=2015,
    max=2023,
    step=1,
    description='Years:'
)

display(year_range)

In [ ]:
# Checkbox
show_trend = widgets.Checkbox(
    value=True,
    description='Show trend line'
)

display(show_trend)

In [ ]:
# Multiple selection
country_select = widgets.SelectMultiple(
    options=list(countries_in_data),
    value=['Germany', 'France'],
    description='Countries:',
    rows=5
)

display(country_select)
print("Hold Ctrl/Cmd to select multiple")

### Connecting Widgets to Visualisations

The `interact` function automatically creates widgets based on function parameters and updates the output when widgets change.

In [ ]:
from ipywidgets import interact, interactive, fixed

def plot_country_trend(country):
    """Plot renewable energy trend for a selected country."""
    country_data = df[df['country'] == country].sort_values('year')
    
    plt.figure(figsize=(10, 5))
    plt.plot(country_data['year'], country_data['renewable_percent'],
             marker='o', linewidth=2, color='teal')
    plt.fill_between(country_data['year'], country_data['renewable_percent'],
                     alpha=0.3, color='teal')
    plt.xlabel('Year')
    plt.ylabel('Renewable Energy %')
    plt.title(f'{country}: Renewable Energy Trend')
    plt.ylim(0, 90)
    plt.tight_layout()
    plt.show()

# Create interactive widget
interact(plot_country_trend, country=list(countries_in_data));

The `interact` function examines the function's parameters and creates appropriate widgets automatically. A list of strings becomes a dropdown, a tuple of numbers becomes a slider, and so on.

In [ ]:
# More complex example with multiple parameters

def plot_comparison(country1, country2, metric, show_difference):
    """Compare two countries on a selected metric."""
    data1 = df[df['country'] == country1].sort_values('year')
    data2 = df[df['country'] == country2].sort_values('year')
    
    # Column name mapping
    column_map = {
        'Renewable %': 'renewable_percent',
        'CO2 Emissions': 'co2_emissions_mt',
        'Energy Consumption': 'energy_consumption_twh'
    }
    col = column_map[metric]
    
    fig, ax = plt.subplots(figsize=(10, 5))
    
    ax.plot(data1['year'], data1[col], marker='o', linewidth=2, 
            label=country1, color='#e74c3c')
    ax.plot(data2['year'], data2[col], marker='s', linewidth=2,
            label=country2, color='#3498db')
    
    if show_difference:
        # Merge on year to calculate difference
        merged = data1[['year', col]].merge(data2[['year', col]], 
                                             on='year', suffixes=('_1', '_2'))
        diff = merged[f'{col}_1'] - merged[f'{col}_2']
        ax.fill_between(merged['year'], merged[f'{col}_1'], merged[f'{col}_2'],
                        alpha=0.2, color='purple', label='Difference')
    
    ax.set_xlabel('Year')
    ax.set_ylabel(metric)
    ax.set_title(f'{country1} vs {country2}: {metric}')
    ax.legend()
    plt.tight_layout()
    plt.show()

interact(plot_comparison,
         country1=list(countries_in_data),
         country2=list(countries_in_data),
         metric=['Renewable %', 'CO2 Emissions', 'Energy Consumption'],
         show_difference=True);

### Manual Widget Layout

For more control over widget arrangement and behaviour, you can create widgets explicitly and use `interactive_output` to connect them to a function.

In [ ]:
# Create widgets with custom styling
style = {'description_width': '100px'}

w_countries = widgets.SelectMultiple(
    options=list(countries_in_data),
    value=['Germany', 'Sweden', 'Poland'],
    description='Countries:',
    style=style,
    rows=6
)

w_year_range = widgets.IntRangeSlider(
    value=[2015, 2023],
    min=2015,
    max=2023,
    description='Year Range:',
    style=style
)

w_chart_type = widgets.RadioButtons(
    options=['Line', 'Bar'],
    value='Line',
    description='Chart Type:',
    style=style
)

def plot_custom(countries, year_range, chart_type):
    """Create customised plot based on widget selections."""
    # Filter data
    mask = (df['country'].isin(countries)) & \
           (df['year'] >= year_range[0]) & \
           (df['year'] <= year_range[1])
    filtered = df[mask]
    
    if len(filtered) == 0:
        print("No data for selection")
        return
    
    fig, ax = plt.subplots(figsize=(10, 5))
    
    if chart_type == 'Line':
        for country in countries:
            cdata = filtered[filtered['country'] == country].sort_values('year')
            ax.plot(cdata['year'], cdata['renewable_percent'],
                    marker='o', linewidth=2, label=country)
    else:  # Bar
        # Get latest year's data
        latest_year = filtered['year'].max()
        bar_data = filtered[filtered['year'] == latest_year]
        ax.bar(bar_data['country'], bar_data['renewable_percent'],
               color='teal', edgecolor='white')
        ax.set_xticklabels(bar_data['country'], rotation=45, ha='right')
    
    ax.set_xlabel('Year' if chart_type == 'Line' else 'Country')
    ax.set_ylabel('Renewable Energy %')
    ax.set_title(f'Renewable Energy ({year_range[0]}-{year_range[1]})')
    if chart_type == 'Line':
        ax.legend()
    plt.tight_layout()
    plt.show()

# Create output widget
out = widgets.interactive_output(plot_custom, {
    'countries': w_countries,
    'year_range': w_year_range,
    'chart_type': w_chart_type
})

# Arrange widgets in a layout
controls = widgets.VBox([w_countries, w_year_range, w_chart_type])
display(widgets.HBox([controls, out]))

The `VBox` and `HBox` widgets arrange their children vertically and horizontally respectively. This gives you control over the dashboard layout. The `interactive_output` function connects the widgets to the plotting function, updating the output whenever any widget changes.

### Buttons and Actions

Buttons let you trigger actions on demand rather than automatically updating. This is useful for operations like saving figures or running calculations that shouldn't happen on every widget change.

In [ ]:
# Output area for messages
output_area = widgets.Output()

# Country selector
save_country = widgets.Dropdown(
    options=list(countries_in_data),
    value='Germany',
    description='Country:'
)

# Save button
save_button = widgets.Button(
    description='Save Figure',
    button_style='primary',  # 'success', 'info', 'warning', 'danger', ''
    icon='save'
)

def on_save_clicked(b):
    """Handle save button click."""
    with output_area:
        output_area.clear_output()
        
        country = save_country.value
        country_data = df[df['country'] == country].sort_values('year')
        
        fig, ax = plt.subplots(figsize=(10, 5))
        ax.plot(country_data['year'], country_data['renewable_percent'],
                marker='o', linewidth=2, color='teal')
        ax.set_xlabel('Year')
        ax.set_ylabel('Renewable Energy %')
        ax.set_title(f'{country}: Renewable Energy Trend')
        
        safe_name = country.lower().replace(' ', '_')
        filename = f'report_figures/{safe_name}_interactive.png'
        fig.savefig(filename, dpi=150, bbox_inches='tight')
        
        plt.show()
        print(f"Saved to: {filename}")

save_button.on_click(on_save_clicked)

display(widgets.HBox([save_country, save_button]))
display(output_area)

## Building a Report

Let's bring everything together by generating a set of figures and summary statistics that could form a report. The approach is to create and save figures programmatically, then provide a summary of what was generated.

In [ ]:
# Report configuration
report_countries = ['Germany', 'France', 'UK', 'Sweden', 'Poland']
report_year = 2023

# Clear the figures directory
import glob
for f in glob.glob('report_figures/*.png'):
    os.remove(f)

report_files = []

In [ ]:
# Figure 1: Overview bar chart
latest = df[(df['year'] == report_year) & (df['country'].isin(report_countries))]
latest = latest.sort_values('renewable_percent', ascending=True)

fig, ax = plt.subplots(figsize=(10, 6))
bars = ax.barh(latest['country'], latest['renewable_percent'], color='teal', edgecolor='white')
ax.set_xlabel('Renewable Energy %', fontsize=12)
ax.set_title(f'Renewable Energy by Country ({report_year})', fontsize=14)

# Add value labels on bars
for bar, val in zip(bars, latest['renewable_percent']):
    ax.text(val + 1, bar.get_y() + bar.get_height()/2, f'{val:.1f}%',
            va='center', fontsize=10)

ax.set_xlim(0, 85)
plt.tight_layout()

filename = 'report_figures/01_overview_bar.png'
fig.savefig(filename, dpi=150, bbox_inches='tight')
report_files.append(filename)
plt.show()
plt.close(fig)

In [ ]:
# Figure 2: Trend comparison
fig, ax = plt.subplots(figsize=(12, 6))

colors = {'Germany': '#000000', 'France': '#3498db', 'UK': '#e74c3c',
          'Sweden': '#f1c40f', 'Poland': '#9b59b6'}

for country in report_countries:
    cdata = df[df['country'] == country].sort_values('year')
    ax.plot(cdata['year'], cdata['renewable_percent'],
            marker='o', linewidth=2, label=country, color=colors.get(country, 'gray'))

ax.set_xlabel('Year', fontsize=12)
ax.set_ylabel('Renewable Energy %', fontsize=12)
ax.set_title('Renewable Energy Trends (2015-2023)', fontsize=14)
ax.legend(loc='upper left')
ax.grid(True, alpha=0.3)

plt.tight_layout()

filename = 'report_figures/02_trends_comparison.png'
fig.savefig(filename, dpi=150, bbox_inches='tight')
report_files.append(filename)
plt.show()
plt.close(fig)

In [ ]:
# Figure 3: Renewables vs Emissions scatter
fig, ax = plt.subplots(figsize=(10, 6))

scatter_data = df[df['year'] == report_year]

for country in scatter_data['country']:
    row = scatter_data[scatter_data['country'] == country].iloc[0]
    color = colors.get(country, 'gray')
    ax.scatter(row['renewable_percent'], row['co2_emissions_mt'],
               s=150, color=color, edgecolor='white', linewidth=2,
               label=country if country in report_countries else None)
    ax.annotate(country, (row['renewable_percent'] + 1, row['co2_emissions_mt']),
                fontsize=9)

ax.set_xlabel('Renewable Energy %', fontsize=12)
ax.set_ylabel('CO2 Emissions (Mt)', fontsize=12)
ax.set_title(f'Renewable Energy vs Emissions ({report_year})', fontsize=14)

plt.tight_layout()

filename = 'report_figures/03_renewables_vs_emissions.png'
fig.savefig(filename, dpi=150, bbox_inches='tight')
report_files.append(filename)
plt.show()
plt.close(fig)

In [ ]:
# Figure 4: Individual country detail pages (loop)
for country in report_countries:
    cdata = df[df['country'] == country].sort_values('year')
    
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    # Left: Renewable trend
    axes[0].plot(cdata['year'], cdata['renewable_percent'],
                 marker='o', linewidth=2, color='teal')
    axes[0].fill_between(cdata['year'], cdata['renewable_percent'],
                         alpha=0.3, color='teal')
    axes[0].set_xlabel('Year')
    axes[0].set_ylabel('Renewable %')
    axes[0].set_title(f'{country}: Renewable Energy')
    axes[0].set_ylim(0, 85)
    
    # Right: Emissions trend
    axes[1].plot(cdata['year'], cdata['co2_emissions_mt'],
                 marker='s', linewidth=2, color='coral')
    axes[1].fill_between(cdata['year'], cdata['co2_emissions_mt'],
                         alpha=0.3, color='coral')
    axes[1].set_xlabel('Year')
    axes[1].set_ylabel('CO2 Emissions (Mt)')
    axes[1].set_title(f'{country}: CO2 Emissions')
    
    fig.suptitle(f'{country} Environmental Profile', fontsize=14, y=1.02)
    plt.tight_layout()
    
    safe_name = country.lower().replace(' ', '_')
    filename = f'report_figures/04_{safe_name}_detail.png'
    fig.savefig(filename, dpi=150, bbox_inches='tight')
    report_files.append(filename)
    plt.close(fig)

print(f"Generated {len(report_files)} figures for report")

In [ ]:
# Summary statistics for the report
summary_data = df[(df['year'] == report_year) & (df['country'].isin(report_countries))]

print("=" * 60)
print(f"RENEWABLE ENERGY REPORT - {report_year}")
print("=" * 60)
print()
print("KEY FINDINGS:")
print()

leader = summary_data.loc[summary_data['renewable_percent'].idxmax()]
print(f"  Highest renewable %: {leader['country']} ({leader['renewable_percent']:.1f}%)")

lowest = summary_data.loc[summary_data['renewable_percent'].idxmin()]
print(f"  Lowest renewable %: {lowest['country']} ({lowest['renewable_percent']:.1f}%)")

print(f"  Average across countries: {summary_data['renewable_percent'].mean():.1f}%")
print()

print("EMISSIONS:")
print()
highest_emit = summary_data.loc[summary_data['co2_emissions_mt'].idxmax()]
print(f"  Highest emitter: {highest_emit['country']} ({highest_emit['co2_emissions_mt']:.1f} Mt)")

lowest_emit = summary_data.loc[summary_data['co2_emissions_mt'].idxmin()]
print(f"  Lowest emitter: {lowest_emit['country']} ({lowest_emit['co2_emissions_mt']:.1f} Mt)")

print()
print("FIGURES GENERATED:")
print()
for f in report_files:
    print(f"  {f}")
print()
print("=" * 60)

## Practice Exercises

**Exercise 1:** Add a new column to the DataFrame called `emissions_trend` that categorises each row as 'Improving' (emissions below the country's mean), 'Stable' (within 10% of mean), or 'Worsening' (above mean by more than 10%). You will need to calculate each country's mean separately and then apply the categorisation.

In [ ]:
# Your code here


**Exercise 2:** Create an interactive widget that lets the user select a year from a slider and displays a bar chart comparing all countries' renewable percentages for that year. Include a checkbox that toggles whether to sort the bars by value or alphabetically by country name.

In [ ]:
# Your code here


**Exercise 3:** Write a loop that creates and saves a histogram of `renewable_percent` values for each year in the dataset (one histogram per year, showing the distribution across countries). Save the figures with filenames that include the year.

In [ ]:
# Your code here


**Exercise 4:** Create a "mini dashboard" using ipywidgets that includes: a dropdown to select a country, a radio button to choose between 'Renewable %' and 'CO2 Emissions', and a button that saves the current chart to a file. The filename should include both the country name and the metric.

In [ ]:
# Your code here


**Exercise 5:** Remove all rows from the DataFrame where `renewable_percent` is below 20, then add three new rows of your own invention representing data for a country not currently in the dataset. Verify your changes by displaying relevant subsets of the DataFrame.

In [ ]:
# Your code here


## Where This Leads

The techniques in this notebook prepare you for more sophisticated data applications. Interactive dashboards can be extended using libraries like Panel, Voila, or Streamlit, which turn notebooks into standalone web applications. For reports, you might explore programmatic document generation with libraries like ReportLab for PDFs or python-pptx for PowerPoint presentations.

The looping patterns for plot generation scale to much larger datasets. If you had data for 200 countries or 50 variables, the same approach would work with minimal modification. This is the practical value of programmatic visualisation: it handles scale gracefully.

For sharing interactive notebooks, Binder (mybinder.org) can turn a GitHub repository into a live, executable notebook environment that anyone can use without installing software. This is particularly useful for educational materials or collaborative analysis where you want others to explore the data themselves.

## Bibliography

ipywidgets documentation. https://ipywidgets.readthedocs.io/ Official documentation with comprehensive widget reference and examples.

VanderPlas, J. (2016). *Python Data Science Handbook*. O'Reilly Media. Chapter 4 covers interactive visualisation. Available free at https://jakevdp.github.io/PythonDataScienceHandbook/

McKinney, W. (2022). *Python for Data Analysis* (3rd ed.). O'Reilly Media. Chapter 9 covers plotting with pandas. https://wesmckinney.com/book/

Real Python: Interactive Data Visualization in Python With Bokeh. https://realpython.com/python-data-visualization-bokeh/ Tutorial on an alternative interactive visualisation library.

Jupyter Widgets documentation. https://jupyter.org/widgets Broader context for widget-based interactivity in Jupyter.

Panel documentation. https://panel.holoviz.org/ For building more sophisticated dashboards from notebooks.

Streamlit documentation. https://docs.streamlit.io/ An alternative approach to creating interactive data applications.

Our World in Data. https://ourworldindata.org/ Source for real environmental datasets to practice with.